# Nucleotide Transformer smoke test

Purpose: check that InstaDeep's repo installs and runs on this Colab runtime (GPU/TPU), and discover the actual output structure of a forward pass — the docs confirm an `embeddings_N` key but do NOT confirm a `logits` key, so this cell finds out empirically rather than assuming.

Set Runtime > Change runtime type > GPU (or TPU) before running.

In [ ]:
!pip install git+https://github.com/instadeepai/nucleotide-transformer.git
!pip install -U "jax[cuda12]" "numpy<2.0" --force-reinstall

**Restart now: Runtime > Restart session.** This cell force-reinstalls jax and numpy — the kernel still has the old binary versions loaded in memory, so nothing after this point is safe to run until you restart. Packages persist on disk across a restart, so do not re-run the cell above afterward — just continue to the next cell.

In [ ]:
import jax
print(jax.devices())

## Load smallest documented-working checkpoint, run one forward pass

Using `"250M_multi_species_v2"` specifically because it's the exact model name shown working in the repo's own docs — not guessing at a smaller variant's name without confirmation. `embeddings_layers_to_save=()` since we don't need embeddings for this check.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import time
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

t0 = time.time()
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name="250M_multi_species_v2",
    embeddings_layers_to_save=(),
    max_positions=32,
)
print(f"load time: {time.time() - t0:.1f}s")

sequences = ["ATTCCGATTCCGATTCCG", "ATTTCTCTCTCTCTCTGAGATCGATCGATCGAT"]
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

random_key = jax.random.PRNGKey(0)
forward_fn = hk.transform(forward_fn)

t0 = time.time()
outs = forward_fn.apply(parameters, random_key, tokens)
print(f"forward pass time: {time.time() - t0:.1f}s")

print(outs.keys())
for k, v in outs.items():
    print(k, v.shape)

## Score engineered-sequence FASTA files (real vs. pseudo)

Upload your FASTA files to this Colab session first (Files pane > upload). Before running: reload the model above with `max_positions` large enough for your longest actual sequence — 32 was only for the smoke test's toy strings.

In [ ]:
import random


class NucleotideTransformerScorer:
    def __init__(self, apply_fn, parameters, tokenizer, mask_fraction=None):
        self.apply_fn = apply_fn
        self.parameters = parameters
        self.tokenizer = tokenizer
        self.mask_fraction = mask_fraction

    def score(self, sequence: str) -> float:
        token_ids = jnp.asarray(self.tokenizer.batch_tokenize([sequence])[0][1])
        special = {self.tokenizer.pad_token_id, self.tokenizer.class_token_id}
        real_positions = [i for i, t in enumerate(token_ids.tolist()) if t not in special]

        positions = real_positions
        if self.mask_fraction is not None:
            k = max(1, int(len(real_positions) * self.mask_fraction))
            positions = random.sample(real_positions, k)

        batch = jnp.stack([token_ids.at[p].set(self.tokenizer.mask_token_id) for p in positions])
        outs = self.apply_fn(self.parameters, jax.random.PRNGKey(0), batch)
        log_probs = jax.nn.log_softmax(outs["logits"], axis=-1)

        true_ids = token_ids[jnp.array(positions)]
        idx = jnp.arange(len(positions))
        scores = log_probs[idx, jnp.array(positions), true_ids]
        return float(jnp.mean(scores))

In [ ]:
!pip install biopython
import re
from Bio import SeqIO


def parse_experience(description):
    match = re.search(r"\[Experience:([^\]]*)\]", description)
    return match.group(1) if match else "Unknown"


def score_igem_fasta(path, scorer):
    rows = []
    for record in SeqIO.parse(path, "fasta"):
        seq = str(record.seq).upper()
        experience = parse_experience(record.description)
        n_count = seq.count("N")
        non_n_length = len(seq) - n_count
        gc = sum(1 for b in seq if b in "GC") / non_n_length * 100 if non_n_length else 0
        score = scorer.score(seq)
        rows.append({"id": record.id, "label": experience, "length": len(seq), "gc_content": gc, "score": score})
        print(record.id, experience, round(score, 3))
    return rows


apply_fn = hk.transform(forward_fn).apply
scorer = NucleotideTransformerScorer(apply_fn, parameters, tokenizer, mask_fraction=0.5)

engineered_rows = score_igem_fasta("data/raw/igem/seq.fasta", scorer)